# подбор параметров для выбранной стратегии. абляции

In [1]:
import os
import time
import copy
import glob
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch_pruning as tp
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

warnings.filterwarnings("ignore")

LOG_DIR = "logs/d05"
os.makedirs(LOG_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Log dir: {os.path.abspath(LOG_DIR)}")

/home/user/projects/Quantization-and-Pruning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Log dir: /home/user/projects/Quantization-and-Pruning/pruning/logs/d05


In [2]:
MODEL_ID = "HuggingFaceTB/SmolLM-135M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_fresh_model():
    """Загружает свежую модель в fp16 на GPU."""
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
    )
    m.to(device)
    m.eval()
    return m


_probe = load_fresh_model()
print(f"Params: {sum(p.numel() for p in _probe.parameters()):,}")
print(f"Heads: {_probe.config.num_attention_heads} | "
      f"KV heads: {_probe.config.num_key_value_heads}")
print(f"Layers: {_probe.config.num_hidden_layers} | "
      f"Intermediate: {_probe.config.intermediate_size}")
del _probe
if device.type == "cuda":
    torch.cuda.empty_cache()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1177.40it/s]


Params: 134,515,008
Heads: 9 | KV heads: 3
Layers: 30 | Intermediate: 1536


In [4]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
test_texts = [t for t in dataset["test"]["text"] if len(t.strip()) > 0]
print(f"Test samples: {len(test_texts)}")


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, texts, max_length=512, stride=256):
    model.eval()
    encodings = tokenizer("\n\n".join(texts), return_tensors="pt")
    input_ids = encodings.input_ids.to(model.device)

    nlls = []
    prev_end = 0
    for begin in range(0, input_ids.size(1), stride):
        end = min(begin + max_length, input_ids.size(1))
        trg_len = end - prev_end

        chunk = input_ids[:, begin:end]
        targets = chunk.clone()
        targets[:, :-trg_len] = -100

        outputs = model(chunk, labels=targets)
        nlls.append(outputs.loss * trg_len)

        prev_end = end
        if end == input_ids.size(1):
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end).item()
    return ppl

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Fri Aug 28 13:04:59 2026).


Test samples: 2891


In [9]:
def calibrate_gradients(model, loader, device, n_batches=3):
    """
    forward+backward на нескольких батчах, чтобы заполнить .grad.
    нужно для gradient-based критериев (Taylor).
    не очищает .grad в конце — pruner.step() должен его прочитать.
    """
    model.eval()  
    model.zero_grad(set_to_none=True)

    n_done = 0
    for input_ids, attention_mask in loader:
        if n_done >= n_batches:
            break
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        outputs.loss.backward()
        n_done += 1

    print(f"    calibrated on {n_done} batches (grads filled)")


def iterative_prune_with_finetune(
    base_model,
    target_ratio,
    steps=5,
    importance="lamp",
    lora_r=8,
    lora_alpha=16,
    finetune_epochs=1,
    finetune_lr=1e-4,
    train_size=2000,
    max_length=512,
    batch_size=2,
):
    m = copy.deepcopy(base_model).to(device)
    m.eval()

    if importance == "taylor":
        imp = tp.importance.TaylorImportance()
    elif importance == "lamp":
        imp = tp.importance.LAMPImportance(p=2)
    elif importance == "magnitude":
        imp = tp.importance.MagnitudeImportance(p=2)
    elif importance == "groupnorm":
        imp = tp.importance.GroupMagnitudeImportance(p=2)
    else:
        raise ValueError(f"Unknown importance: {importance}")

    # --- Ignored layers: GQA + выход ---
    ignored = [m.lm_head, m.model.embed_tokens]
    if m.config.num_attention_heads != m.config.num_key_value_heads:
        for layer in m.model.layers:
            ignored.extend([
                layer.self_attn.q_proj,
                layer.self_attn.k_proj,
                layer.self_attn.v_proj,
                layer.self_attn.o_proj,
            ])

    example_inputs = tokenizer("Hello world", return_tensors="pt").to(device)

    pruner = tp.pruner.MagnitudePruner(
        m, example_inputs,
        importance=imp,
        global_pruning=True,
        pruning_ratio=target_ratio,
        iterative_steps=steps,
        ignored_layers=ignored,
        forward_fn=lambda model, inputs: model(**inputs).logits,
    )

    train_texts = [t for t in dataset["train"]["text"] if len(t.strip()) > 50][:train_size]
    encodings = tokenizer(
        train_texts, truncation=True, max_length=max_length,
        padding="max_length", return_tensors="pt",
    )
    train_ds = torch.utils.data.TensorDataset(
        encodings["input_ids"], encodings["attention_mask"]
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    for step in range(steps):
        if importance == "taylor":
            calibrate_gradients(m, train_loader, device, n_batches=3)

        pruner.step()

        m.zero_grad(set_to_none=True)

        print(f"    step {step+1}/{steps}: params={sum(p.numel() for p in m.parameters()):,}")

        lora_config = LoraConfig(
            r=lora_r, lora_alpha=lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        )
        peft_model = get_peft_model(m, lora_config)
        peft_model.train()

        optimizer = torch.optim.AdamW(peft_model.parameters(), lr=finetune_lr)
        total_steps = len(train_loader) * finetune_epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=max(1, total_steps // 10),
            num_training_steps=total_steps,
        )

        for _ in range(finetune_epochs):
            for input_ids, attention_mask in train_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = input_ids.clone()
                labels[attention_mask == 0] = -100

                outputs = peft_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        m = peft_model.merge_and_unload()
        m.eval()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return m

In [10]:
# базовый сетап
DEFAULT_CONFIG = {
    "importance":      "lamp",
    "steps":           5,
    "finetune_epochs": 2,
    "lora_r":          8,
    "lora_alpha":      16,
    "finetune_lr":     1e-4,
    "train_size":      2000,
    "target_ratio":    0.05,
    "max_length":      512,
    "batch_size":      2,
}

# сетка значений для каждой группы (варьируется только этот параметр)
ABLATION_GROUPS = {
    "importance":      ["lamp", "taylor", "magnitude"],
    "steps":           [3, 5, 10],
    "finetune_epochs": [1, 2, 3],
    "lora_r":          [8, 16, 32],
    "target_ratio":    [0.02, 0.05, 0.10],
    "train_size":      [1000, 2000, 5000],
    "finetune_lr":     [5e-5, 1e-4, 2e-4],
}

ACTIVE_GROUPS = [
    "importance",
    "target_ratio",
    # "steps",
    # "finetune_epochs",
    # "lora_r",
    # "train_size",
    # "finetune_lr",
]

SKIP_EXISTING = True

n_runs = sum(len(ABLATION_GROUPS[g]) for g in ACTIVE_GROUPS)
print(f"Active groups: {ACTIVE_GROUPS}")
print(f"Total runs: {n_runs}")
print(f"Skip existing CSVs: {SKIP_EXISTING}")

Active groups: ['importance', 'target_ratio']
Total runs: 6
Skip existing CSVs: True


In [11]:
def run_single_ablation(group_name, value, config):
    exp_name = f"{group_name}={value}"
    print(f"\n{'='*60}\nExperiment: {exp_name}\n{'='*60}")

    t_start = time.perf_counter()

    try:
        base = load_fresh_model()
        pruned = iterative_prune_with_finetune(
            base,
            target_ratio=config["target_ratio"],
            steps=config["steps"],
            importance=config["importance"],
            lora_r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            finetune_epochs=config["finetune_epochs"],
            finetune_lr=config["finetune_lr"],
            train_size=config["train_size"],
            max_length=config["max_length"],
            batch_size=config["batch_size"],
        )
        del base
        if device.type == "cuda":
            torch.cuda.empty_cache()

        n_params = sum(p.numel() for p in pruned.parameters())
        model_size_mb = n_params * 2 / (1024 ** 2)

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        t_eval = time.perf_counter()
        ppl = evaluate_perplexity(pruned, tokenizer, test_texts)
        eval_time_s = time.perf_counter() - t_eval

        peak_mem_mb = (
            torch.cuda.max_memory_allocated() / (1024 ** 2)
            if device.type == "cuda" else 0.0
        )

        result = {
            "exp_name": exp_name,
            "ablation_group": group_name,
            "ablation_value": str(value),
            **config,
            "params": n_params,
            "model_size_mb": model_size_mb,
            "peak_mem_mb": peak_mem_mb,
            "ppl": ppl,
            "eval_time_s": eval_time_s,
            "total_time_s": time.perf_counter() - t_start,
            "error": "",
        }

        print(f"  → PPL={ppl:.2f} | params={n_params:,} | "
              f"size={model_size_mb:.1f}MB | total={result['total_time_s']:.0f}s")

        del pruned
        if device.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"  → FAILED: {type(e).__name__}: {e}")
        result = {
            "exp_name": exp_name,
            "ablation_group": group_name,
            "ablation_value": str(value),
            **config,
            "params": np.nan, "model_size_mb": np.nan,
            "peak_mem_mb": np.nan, "ppl": np.nan,
            "eval_time_s": np.nan,
            "total_time_s": time.perf_counter() - t_start,
            "error": f"{type(e).__name__}: {e}",
        }
        if device.type == "cuda":
            torch.cuda.empty_cache()

    out_path = os.path.join(LOG_DIR, f"{exp_name}.csv")
    pd.DataFrame([result]).to_csv(out_path, index=False)
    return result

In [ ]:
results = []

for group_name in ACTIVE_GROUPS:
    values = ABLATION_GROUPS[group_name]
    print(f"\n{'#'*60}\n### Group: {group_name} = {values}\n{'#'*60}")

    for value in values:
        exp_name = f"{group_name}={value}"
        out_path = os.path.join(LOG_DIR, f"{exp_name}.csv")

        if SKIP_EXISTING and os.path.exists(out_path):
            print(f"\n[SKIP] {exp_name} — already exists")
            results.append(pd.read_csv(out_path).iloc[0].to_dict())
            continue

        config = DEFAULT_CONFIG.copy()
        config[group_name] = value
        results.append(run_single_ablation(group_name, value, config))

print(f"\n{'='*60}\nDONE: {len(results)} experiments\n{'='*60}")


############################################################
### Group: importance = ['lamp', 'taylor', 'magnitude']
############################################################

Experiment: importance=lamp


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4059.95it/s]


    step 1/5: params=133,718,400
    step 2/5: params=132,921,792
    step 3/5: params=132,125,184
    step 4/5: params=131,328,576
    step 5/5: params=130,533,696
  → PPL=25.60 | params=130,533,696 | size=249.0MB | total=860s

Experiment: importance=taylor


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 3393.06it/s]


    calibrated on 3 batches (grads filled)
    step 1/5: params=133,718,400
  → FAILED: RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

Experiment: importance=magnitude


...

    step 1/5: params=134,195,328
    step 2/5: params=133,877,376
    step 3/5: params=133,559,424
    step 4/5: params=133,239,744
    step 5/5: params=132,921,792
  → PPL=21.85 | params=132,921,792 | size=253.5MB | total=853s

Experiment: target_ratio=0.05


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 3915.48it/s]


    step 1/5: params=133,718,400
    step 2/5: params=132,921,792
    step 3/5: params=132,125,184
    step 4/5: params=131,328,576
    step 5/5: params=130,533,696
  → PPL=25.53 | params=130,533,696 | size=249.0MB | total=851s

Experiment: target_ratio=0.1


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4052.48it/s]


    step 1/5: params=132,921,792
    step 2/5: params=131,328,576
    step 3/5: params=129,737,088
    step 4/5: params=128,143,872


In [ ]:
paths = sorted(glob.glob(os.path.join(LOG_DIR, "*.csv")))
if not paths:
    raise RuntimeError(f"Нет результатов в {LOG_DIR}. Запустите эксперимент.")

df_all = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

show_cols = ["exp_name", "ppl", "params", "model_size_mb",
             "peak_mem_mb", "total_time_s", "error"]
print("=== ALL RESULTS ===")
print(df_all[show_cols].to_string(index=False))

success = df_all[(df_all["error"].isna()) | (df_all["error"] == "")].copy()
failed  = df_all.drop(success.index).copy()

print(f"\nSuccessful: {len(success)} | Failed: {len(failed)}")

for group in success["ablation_group"].unique():
    sub = success[success["ablation_group"] == group].copy()
    order = {str(v): i for i, v in enumerate(ABLATION_GROUPS.get(group, []))}
    sub["_order"] = sub["ablation_value"].map(order)
    sub = sub.sort_values("_order")

    best = sub.loc[sub["ppl"].idxmin()]
    print(f"\n[{group}]")
    print(f"  Best: {best['exp_name']}  →  PPL={best['ppl']:.2f}  "
          f"params={int(best['params']):,}  size={best['model_size_mb']:.1f} MB")
    print(sub[["ablation_value", "ppl", "params", "total_time_s"]].to_string(index=False))

if len(failed):
    print("\n=== FAILED RUNS ===")
    print(failed[["exp_name", "error"]].to_string(index=False))

groups = list(success["ablation_group"].unique())
n_groups = len(groups)

fig, axes = plt.subplots(1, n_groups, figsize=(5.5 * n_groups, 5.2))
if n_groups == 1:
    axes = [axes]

for ax, group in zip(axes, groups):
    sub = success[success["ablation_group"] == group].copy()
    order = {str(v): i for i, v in enumerate(ABLATION_GROUPS.get(group, []))}
    sub["_order"] = sub["ablation_value"].map(order)
    sub = sub.sort_values("_order").reset_index(drop=True)

    x = range(len(sub))
    bars = ax.bar(x, sub["ppl"].values, color="tab:blue", alpha=0.85)

    best_idx = int(sub["ppl"].idxmin())
    bars[best_idx].set_color("tab:green")

    ax.set_xticks(list(x))
    ax.set_xticklabels(sub["ablation_value"].values, rotation=30, ha="right")
    ax.set_xlabel(group)
    ax.set_ylabel("PPL (log scale, lower is better)")
    ax.set_title(f"Ablation: {group}")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3, which="both", axis="y")

    for bar, ppl in zip(bars, sub["ppl"].values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.15,
                f"{ppl:.1f}", ha="center", fontsize=9)

plt.suptitle(
    f"Ablation summary (default ratio={DEFAULT_CONFIG['target_ratio']}, "
    f"importance={DEFAULT_CONFIG['importance']})",
    fontsize=13, y=1.02,
)
plt.tight_layout()

out_png = os.path.join(LOG_DIR, "ablation_summary.png")
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✓ Saved: {out_png}")

if len(success):
    best_overall = success.loc[success["ppl"].idxmin()]
    print(f"\n{'='*60}\nBEST OVERALL CONFIG\n{'='*60}")
    for k in ["importance", "steps", "finetune_epochs", "lora_r",
              "lora_alpha", "finetune_lr", "train_size", "target_ratio"]:
        if k in best_overall:
            print(f"  {k:20s}: {best_overall[k]}")
    print(f"  {'PPL':20s}: {best_overall['ppl']:.2f}")
    print(f"  {'params':20s}: {int(best_overall['params']):,}")
    print(f"  {'size_mb':20s}: {best_overall['model_size_mb']:.1f}")